# DHd Blog Web Scraper

## 1. Environment Setup


In [1]:
# Standard-library imports
import re
import time
from pathlib import Path
from urllib.parse import parse_qs, urlencode, urljoin, urlparse, urlunparse

# Third-party imports
import pandas as pd
import requests
from bs4 import BeautifulSoup
from IPython.display import display
from tqdm.auto import tqdm


# ============================================================
# PORTABLE PROJECT PATHS
# ============================================================
# In a Python script, use the script's folder.
# In Jupyter/Colab, use the current working folder.
try:
    CURRENT_DIR = Path(__file__).resolve().parent
except NameError:
    CURRENT_DIR = Path.cwd().resolve()

# When the notebook is stored in a "notebooks" folder, use its parent
# as the repository root. In Colab or a standalone folder, use the
# current directory directly.
PROJECT_ROOT = (
    CURRENT_DIR.parent
    if CURRENT_DIR.name.lower() == "notebooks"
    else CURRENT_DIR
)

DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

BASE_URL = "https://dhd-blog.org/"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0 Safari/537.36 "
        "Academic DHd Blog analysis"
    )
}

session = requests.Session()
session.headers.update(HEADERS)

GERMAN_MONTHS = (
    "Januar|Februar|März|April|Mai|Juni|Juli|"
    "August|September|Oktober|November|Dezember"
)


def get_soup(url, retries=4, timeout=40):
    """Download one webpage and return a BeautifulSoup object."""

    last_error = None

    for attempt in range(retries):
        try:
            response = session.get(url, timeout=timeout)
            response.raise_for_status()

            return BeautifulSoup(response.content, "lxml")

        except requests.RequestException as error:
            last_error = error

            if attempt < retries - 1:
                wait_time = 2 ** attempt
                time.sleep(wait_time)

    raise last_error


def clean_spacing(text):
    """Normalize spaces while keeping paragraph breaks."""

    if not text:
        return ""

    text = text.replace("\xa0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


def get_post_id(url):
    """Return the numeric WordPress post ID from a ?p= URL."""

    query = parse_qs(urlparse(url).query)
    post_id = query.get("p", [""])[0]

    return str(post_id) if str(post_id).isdigit() else ""


def normalize_post_url(url):
    """Standardize post URLs to avoid duplicates."""

    post_id = get_post_id(url)

    if not post_id:
        return ""

    return f"{BASE_URL}?p={post_id}"


print("Setup complete.")

Setup complete.


c:\Users\jiwani\Documents\dhd-blog-analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Collect Archive Pages and Post Links


In [2]:
def get_archive_months():
    """Extract all monthly archive links from the website."""

    soup = get_soup(BASE_URL)
    months = {}

    for link in soup.find_all("a", href=True):
        label = clean_spacing(link.get_text(" ", strip=True))

        if re.fullmatch(
            rf"(?:{GERMAN_MONTHS})\s+\d{{4}}",
            label
        ):
            archive_url = urljoin(BASE_URL, link["href"])
            months[label] = archive_url

    return pd.DataFrame(
        [
            {
                "archive_month": month,
                "archive_url": url
            }
            for month, url in months.items()
        ]
    )


def build_archive_page_url(archive_url, page_number):
    """
    Build page 1, page 2, page 3, etc. for one archive month.

    Example:
    ?m=202503
    ?m=202503&paged=2
    ?m=202503&paged=3
    """

    parsed = urlparse(archive_url)
    query = parse_qs(parsed.query)

    if page_number <= 1:
        query.pop("paged", None)
    else:
        query["paged"] = [str(page_number)]

    new_query = urlencode(query, doseq=True)

    return urlunparse(
        (
            parsed.scheme,
            parsed.netloc,
            parsed.path,
            parsed.params,
            new_query,
            ""
        )
    )


def extract_post_links_from_archive(soup):
    """Extract all individual post links from one archive page."""

    post_urls = set()

    selectors = [
        "h2 a[rel='bookmark'][href]",
        "#content h2 a[href]",
        "article h2 a[href]",
        "article h1 a[href]"
    ]

    for selector in selectors:
        for link in soup.select(selector):
            full_url = urljoin(BASE_URL, link["href"])
            normalized_url = normalize_post_url(full_url)

            if normalized_url:
                post_urls.add(normalized_url)

    # Fallback for older HTML layouts
    if not post_urls:
        for heading in soup.find_all(["h1", "h2"]):
            link = heading.find("a", href=True)

            if not link:
                continue

            full_url = urljoin(BASE_URL, link["href"])
            normalized_url = normalize_post_url(full_url)

            if normalized_url:
                post_urls.add(normalized_url)

    return post_urls


archive_df = get_archive_months()

print("Archive months found:", len(archive_df))

post_records = []
archive_errors = []
global_seen_post_ids = set()

# Safety limit only prevents an accidental infinite loop.
# A month can have up to 100 pages before this stops.
MAX_PAGES_PER_MONTH = 100

for _, archive in tqdm(
    archive_df.iterrows(),
    total=len(archive_df),
    desc="Collecting all archive pages"
):
    month_seen_ids = set()
    empty_or_repeated_pages = 0

    for page_number in range(1, MAX_PAGES_PER_MONTH + 1):

        page_url = build_archive_page_url(
            archive["archive_url"],
            page_number
        )

        try:
            soup = get_soup(page_url)
            page_post_urls = extract_post_links_from_archive(soup)

        except Exception as error:
            archive_errors.append(
                {
                    "archive_month": archive["archive_month"],
                    "archive_page": page_number,
                    "archive_page_url": page_url,
                    "error": str(error)
                }
            )

            print(
                f"\nError: {archive['archive_month']} "
                f"page {page_number}: {error}"
            )
            break

        page_post_ids = {
            get_post_id(url)
            for url in page_post_urls
            if get_post_id(url)
        }

        new_month_post_ids = page_post_ids - month_seen_ids

        # Stop when the page is empty or repeats an earlier page.
        if not page_post_ids or not new_month_post_ids:
            empty_or_repeated_pages += 1

            if empty_or_repeated_pages >= 1:
                break
        else:
            empty_or_repeated_pages = 0

        for post_url in page_post_urls:
            post_id = get_post_id(post_url)

            if not post_id:
                continue

            month_seen_ids.add(post_id)

            if post_id not in global_seen_post_ids:
                global_seen_post_ids.add(post_id)

                post_records.append(
                    {
                        "post_id": post_id,
                        "url": post_url,
                        "archive_month": archive["archive_month"],
                        "archive_page": page_number,
                        "archive_page_url": page_url
                    }
                )

        time.sleep(0.4)


post_links_df = pd.DataFrame(post_records)
archive_errors_df = pd.DataFrame(archive_errors)

if post_links_df.empty:
    raise RuntimeError(
        "No posts were found. Stop and check the archive-page HTML."
    )

print("\nUnique posts found:", len(post_links_df))
print("Highest archive page found:", post_links_df["archive_page"].max())

print("\nPosts collected by archive page:")
display(
    post_links_df["archive_page"]
    .value_counts()
    .sort_index()
    .rename_axis("archive_page")
    .reset_index(name="posts")
)

print("\nMonths containing more than one archive page:")
display(
    post_links_df.loc[
        post_links_df["archive_page"] > 1,
        ["archive_month", "archive_page"]
    ]
    .drop_duplicates()
    .sort_values(["archive_month", "archive_page"])
    .head(50)
)

display(post_links_df.head())

Archive months found: 176



Error: Juli 2026 page 4: 404 Client Error: Not Found for url: https://dhd-blog.org/?m=202607&paged=4



Error: Juni 2026 page 3: 404 Client Error: Not Found for url: https://dhd-blog.org/?m=202606&paged=3



Error: Mai 2026 page 2: 404 Client Error: Not Found for url: https://dhd-blog.org/?m=202605&paged=2



Error: April 2026 page 4: 404 Client Error: Not Found for url: https://dhd-blog.org/?m=202604&paged=4



Error: März 2026 page 5: 404 Client Error: Not Found for url: https://dhd-blog.org/?m=202603&paged=5


... (171 outputs left out)

,archive_page,posts
0,1,1707
1,2,898
2,3,123
3,4,23
4,5,1



Months containing more than one archive page:


,archive_month,archive_page
2709,April 2012,2
2593,April 2013,2
2385,April 2014,2
1632,April 2018,2
1429,April 2019,2
1439,April 2019,3
977,April 2021,2
987,April 2021,3
760,April 2022,2
589,April 2023,2


,post_id,url,archive_month,archive_page,archive_page_url
0,23968,https://dhd-blog.org/?p=23968,Juli 2026,1,https://dhd-blog.org/?m=202607
1,23987,https://dhd-blog.org/?p=23987,Juli 2026,1,https://dhd-blog.org/?m=202607
2,23978,https://dhd-blog.org/?p=23978,Juli 2026,1,https://dhd-blog.org/?m=202607
3,23972,https://dhd-blog.org/?p=23972,Juli 2026,1,https://dhd-blog.org/?m=202607
4,23954,https://dhd-blog.org/?p=23954,Juli 2026,1,https://dhd-blog.org/?m=202607


In [3]:
print(post_links_df["archive_page"].max())
print(post_links_df["archive_page"].value_counts().sort_index())

5
archive_page
1    1707
2     898
3     123
4      23
5       1
Name: count, dtype: int64


In [4]:
print("Total collected rows:", len(post_links_df))
print("Unique post IDs:", post_links_df["post_id"].nunique())
print("Duplicate rows:", post_links_df["post_id"].duplicated().sum())

Total collected rows: 2752
Unique post IDs: 2752
Duplicate rows: 0


## 3. Extract and Clean Blog Post Content


In [5]:
# ============================================================
# FULL DHd BLOG POST SCRAPER
# Uses the already-created post_links_df with all archive pages
# ============================================================

def find_article(soup):
    """Locate the main article container."""

    selectors = [
        "#content article.article",
        "#content article",
        "div#content article",
        "article.article",
        "article"
    ]

    for selector in selectors:
        article = soup.select_one(selector)

        if article is not None:
            return article

    return None


def first_matching_text(container, selectors):
    """Return cleaned text from the first matching element."""

    for selector in selectors:
        element = container.select_one(selector)

        if element:
            text = clean_spacing(
                element.get_text(" ", strip=True)
            )

            if text:
                return text

    return ""


def extract_title(article):
    """Extract the blog-post title."""

    return first_matching_text(
        article,
        [
            "h1.entry-title",
            "h1 a[rel='bookmark']",
            "h1",
            "h2.entry-title",
            "h2 a[rel='bookmark']"
        ]
    )


def extract_author(article):
    """Extract the post author."""

    author = first_matching_text(
        article,
        [
            "a[rel='author']",
            ".author a",
            ".byline a",
            ".vcard a"
        ]
    )

    if author:
        return author

    article_text = article.get_text(" ", strip=True)

    match = re.search(
        r"Veröffentlicht\s+von\s+(.+?)\s+am\s+"
        r"\d{1,2}\.\s+\w+\s+\d{4}",
        article_text,
        flags=re.IGNORECASE
    )

    return clean_spacing(match.group(1)) if match else ""


def extract_publication_date(article):
    """Return the publication date as YYYY-MM-DD where possible."""

    time_element = article.select_one("time[datetime]")

    if time_element:
        datetime_value = time_element.get("datetime", "")

        match = re.search(
            r"\d{4}-\d{2}-\d{2}",
            datetime_value
        )

        if match:
            return match.group(0)

    article_text = article.get_text(" ", strip=True)

    match = re.search(
        rf"(\d{{1,2}}\.\s+(?:{GERMAN_MONTHS})\s+\d{{4}})",
        article_text,
        flags=re.IGNORECASE
    )

    if not match:
        return ""

    month_numbers = {
        "januar": 1,
        "februar": 2,
        "märz": 3,
        "april": 4,
        "mai": 5,
        "juni": 6,
        "juli": 7,
        "august": 8,
        "september": 9,
        "oktober": 10,
        "november": 11,
        "dezember": 12
    }

    parts = match.group(1).replace(".", "").split()

    if len(parts) != 3:
        return match.group(1)

    day = int(parts[0])
    month = month_numbers.get(parts[1].lower())
    year = int(parts[2])

    if month is None:
        return match.group(1)

    return f"{year:04d}-{month:02d}-{day:02d}"


def unique_clean_values(values):
    """Remove empty and duplicate values while preserving order."""

    output = []

    for value in values:
        value = clean_spacing(value)

        if value and value not in output:
            output.append(value)

    return output


def extract_categories(article):
    """Extract WordPress categories only."""

    categories = []

    for link in article.find_all("a", href=True):
        href = urljoin(BASE_URL, link["href"])
        parsed = urlparse(href)

        rel_values = link.get("rel", [])

        if isinstance(rel_values, str):
            rel_values = rel_values.split()

        is_category = (
            "/category/" in parsed.path.lower()
            or "category" in rel_values
        )

        if is_category:
            categories.append(
                link.get_text(" ", strip=True)
            )

    # Fallback for category containers
    for link in article.select(
        ".cat-links a, "
        ".post-categories a, "
        ".entry-categories a"
    ):
        categories.append(
            link.get_text(" ", strip=True)
        )

    return unique_clean_values(categories)


def extract_tags(article):
    """Extract WordPress tags only and exclude category links."""

    tags = []

    for link in article.find_all("a", href=True):
        href = urljoin(BASE_URL, link["href"])
        parsed = urlparse(href)

        rel_values = link.get("rel", [])

        if isinstance(rel_values, str):
            rel_values = rel_values.split()

        is_category = (
            "/category/" in parsed.path.lower()
            or "category" in rel_values
        )

        is_tag = (
            "/tag/" in parsed.path.lower()
            or (
                "tag" in rel_values
                and not is_category
            )
        )

        if is_tag and not is_category:
            tags.append(
                link.get_text(" ", strip=True)
            )

    # Fallback for tag containers
    for link in article.select(
        ".tags-links a, "
        ".post-tags a, "
        ".entry-tags a"
    ):
        href = urljoin(
            BASE_URL,
            link.get("href", "")
        )

        if "/category/" not in urlparse(href).path.lower():
            tags.append(
                link.get_text(" ", strip=True)
            )

    return unique_clean_values(tags)


def extract_comment_count(soup, article):
    """Extract the number of comments."""

    for selector in [
        ".comments-link",
        ".comment-count",
        "a[href*='#comments']"
    ]:
        element = article.select_one(selector)

        if element:
            match = re.search(
                r"\d+",
                element.get_text(" ", strip=True)
            )

            if match:
                return int(match.group())

    comments = soup.select(
        "#comments li.comment, "
        ".comment-list li.comment, "
        "ol.commentlist li.comment"
    )

    return len(comments)


def extract_clean_article_text(article, title):
    """
    Extract only the actual blog-post body.

    Removes:
    - title
    - author/date metadata
    - categories
    - tags
    - comments
    - sharing controls
    - navigation
    - related-post sections
    """

    cleaned = BeautifulSoup(
        str(article),
        "lxml"
    )

    remove_selectors = [
        "script",
        "style",
        "noscript",
        "iframe",
        "form",
        "nav",
        "header",
        "footer",

        "h1.entry-title",
        "h2.entry-title",
        "h1 > a[rel='bookmark']",
        "h2 > a[rel='bookmark']",

        ".entry-meta",
        ".post-meta",
        ".postmetadata",
        ".post-meta-data",
        ".entry-footer",
        ".post-footer",
        ".meta",

        ".byline",
        ".author",
        ".vcard",
        ".date",
        ".published",

        ".cat-links",
        ".tags-links",
        ".post-categories",
        ".entry-categories",
        ".post-tags",
        ".entry-tags",

        ".comments-link",
        ".comment-count",
        ".sharedaddy",
        ".sd-sharing-enabled",
        ".share-buttons",
        ".social-share",
        ".navigation",
        ".post-navigation",
        ".comments-area",
        "#comments",
        "#respond",
        ".leave-comment",
        ".comment-respond",
        ".related-posts",
        ".jp-relatedposts",
        ".yarpp-related"
    ]

    for selector in remove_selectors:
        for element in cleaned.select(selector):
            element.decompose()

    # Remove remaining category/tag metadata blocks
    metadata_links = cleaned.select(
        "a[href*='/category/'], "
        "a[href*='/tag/'], "
        "a[rel~='category'], "
        "a[rel~='tag']"
    )

    for link in metadata_links:
        parent = link.parent

        if parent:
            parent_text = clean_spacing(
                parent.get_text(" ", strip=True)
            ).lower()

            metadata_terms = [
                "kategorie",
                "kategorien",
                "category",
                "categories",
                "schlagwort",
                "schlagwörter",
                "tags"
            ]

            if any(
                term in parent_text
                for term in metadata_terms
            ):
                parent.decompose()
            else:
                link.decompose()

    text = cleaned.get_text(
        separator="\n",
        strip=True
    )

    text = clean_spacing(text)

    # Remove title if it remains at the beginning
    if title and text.lower().startswith(title.lower()):
        text = text[len(title):].strip()

    # Remove remaining author/date metadata line
    text = re.sub(
        r"^\s*\d*\s*Veröffentlicht\s+von\s+.+?"
        r"\s+am\s+\d{1,2}\.\s+\w+\s+\d{4}\s*",
        "",
        text,
        flags=re.IGNORECASE
    )

    # Remove remaining category/tag/comment footer
    footer_patterns = [
        r"\n\s*Kategorien?\s*:.*$",
        r"\n\s*Categories?\s*:.*$",
        r"\n\s*Schlagwörter?\s*:.*$",
        r"\n\s*Tags?\s*:.*$",
        r"\n\s*Kommentar schreiben.*$",
        r"\n\s*Hinterlasse einen Kommentar.*$"
    ]

    for pattern in footer_patterns:
        text = re.sub(
            pattern,
            "",
            text,
            flags=re.IGNORECASE | re.DOTALL
        )

    return clean_spacing(text)


def count_words(text):
    """Count words only in the cleaned article body."""

    if not text:
        return 0

    words = re.findall(
        r"\b[0-9A-Za-zÀ-ÖØ-öø-ÿÄÖÜäöüß]+"
        r"(?:[-’'][0-9A-Za-zÀ-ÖØ-öø-ÿÄÖÜäöüß]+)*\b",
        text,
        flags=re.UNICODE
    )

    return len(words)


def scrape_post(record):
    """Scrape one complete DHd Blog post."""

    soup = get_soup(record["url"])
    article = find_article(soup)

    if article is None:
        raise ValueError("Main article element not found")

    title = extract_title(article)
    publication_date = extract_publication_date(article)
    author = extract_author(article)
    categories = extract_categories(article)
    tags = extract_tags(article)

    article_text = extract_clean_article_text(
        article,
        title
    )

    parsed_date = pd.to_datetime(
        publication_date,
        errors="coerce"
    )

    return {
        "post_id": record["post_id"],
        "url": record["url"],
        "archive_month": record["archive_month"],
        "archive_page": record["archive_page"],
        "archive_page_url": record["archive_page_url"],

        "title": title,
        "publication_date": publication_date,

        "year": (
            int(parsed_date.year)
            if not pd.isna(parsed_date)
            else None
        ),

        "month": (
            int(parsed_date.month)
            if not pd.isna(parsed_date)
            else None
        ),

        "author": author,

        "categories": " | ".join(categories),
        "category_count": len(categories),

        "tags": " | ".join(tags),
        "tag_count": len(tags),

        "comments": extract_comment_count(
            soup,
            article
        ),

        "word_count": count_words(article_text),
        "text": article_text
    }


# ============================================================
# START A COMPLETELY NEW FULL RUN
# The previous 10-test rows are discarded here.
# ============================================================

rows = []
post_errors = []

# Use every post exactly once
records = (
    post_links_df
    .drop_duplicates(subset=["post_id"])
    .reset_index(drop=True)
    .to_dict("records")
)

print("Posts to scrape:", len(records))


for index, record in enumerate(
    tqdm(
        records,
        desc="Scraping all DHd Blog posts"
    ),
    start=1
):
    try:
        rows.append(
            scrape_post(record)
        )

    except Exception as error:
        post_errors.append(
            {
                "post_id": record["post_id"],
                "url": record["url"],
                "archive_month": record["archive_month"],
                "archive_page": record["archive_page"],
                "error": str(error)
            }
        )

    # Save a recovery checkpoint every 100 processed posts
    if index % 100 == 0:
        checkpoint_df = pd.DataFrame(rows)

        checkpoint_df.to_csv(
            "dhd_blog_checkpoint.csv",
            index=False,
            encoding="utf-8-sig"
        )

        pd.DataFrame(post_errors).to_csv(
            "dhd_blog_checkpoint_errors.csv",
            index=False,
            encoding="utf-8-sig"
        )

    time.sleep(0.4)


# Create final DataFrames
df = pd.DataFrame(rows)
post_errors_df = pd.DataFrame(post_errors)


if df.empty:
    raise RuntimeError(
        "No posts were successfully scraped."
    )


# Remove any accidental duplicates
df = (
    df.drop_duplicates(
        subset=["post_id"],
        keep="first"
    )
    .reset_index(drop=True)
)


print("\nScraping completed.")
print("Successfully scraped:", len(df))
print("Unique post IDs:", df["post_id"].nunique())
print("Duplicate rows:", df["post_id"].duplicated().sum())
print("Post errors:", len(post_errors_df))


# ============================================================
# VALIDATION CHECKS
# ============================================================

print(
    "\nMissing titles:",
    df["title"].fillna("").str.strip().eq("").sum()
)

print(
    "Missing dates:",
    df["publication_date"].fillna("").str.strip().eq("").sum()
)

print(
    "Missing authors:",
    df["author"].fillna("").str.strip().eq("").sum()
)

print(
    "Empty texts:",
    df["text"].fillna("").str.strip().eq("").sum()
)


df["text_starts_with_title"] = df.apply(
    lambda row: (
        bool(row["title"])
        and bool(row["text"])
        and row["text"].strip().lower().startswith(
            row["title"].strip().lower()
        )
    ),
    axis=1
)


print(
    "Texts starting with title:",
    df["text_starts_with_title"].sum()
)

print(
    "Posts with categories:",
    df["categories"]
    .fillna("")
    .str.strip()
    .ne("")
    .sum()
)

print(
    "Posts with tags:",
    df["tags"]
    .fillna("")
    .str.strip()
    .ne("")
    .sum()
)

print(
    "Posts from archive page 2 or later:",
    (df["archive_page"] >= 2).sum()
)

print(
    "Highest archive page included:",
    df["archive_page"].max()
)


# Show a sample
display(
    df[
        [
            "title",
            "publication_date",
            "author",
            "archive_month",
            "archive_page",
            "categories",
            "tags",
            "comments",
            "word_count",
            "text"
        ]
    ].head(10)
)

Posts to scrape: 2752


Scraping all DHd Blog posts: 100%|██████████| 2752/2752 [23:51<00:00,  1.92it/s]


Scraping completed.
Successfully scraped: 2750
Unique post IDs: 2750
Duplicate rows: 0
Post errors: 2

Missing titles: 0
Missing dates: 0
Missing authors: 0
Empty texts: 0
Texts starting with title: 0
Posts with categories: 2748
Posts with tags: 2036
Posts from archive page 2 or later: 1045
Highest archive page included: 5


,title,publication_date,author,archive_month,archive_page,categories,tags,comments,word_count,text
0,Aufeinander zugehen: Die Dokumentation des GND...,2026-07-27,Anja Gerber,Juli 2026,1,Allgemein | Forschungsdaten | Forschungsinfras...,Dokumentation | Fishbowl-Diskussion | GND | NF...,0,177,Wie kann die Gemeinsame Normdatei (GND) die Zu...
1,30. AWK: Data Ethics for Archival Science and ...,2026-07-28,Robert Meier,Juli 2026,1,Allgemein,Datenethik,0,205,Im Ausland ist Datenethik bereits ein großes T...
2,Professur für Bibliotheks- und Informationsmar...,2026-07-27,Mirjam Blümm,Juli 2026,1,Stellenanzeigen,Bibliotheks- und Informationswissenschaft | TH...,0,179,an der TH Köln ist eine Professur für „Bibliot...
3,Call for proposals: NFDI4Culture Art History A...,2026-07-27,Anja Gerber,Juli 2026,1,Allgemein | Call,Arbeitskreis Digitale Kunstgeschichte | Datenp...,0,482,Der NFDI4Culture Art History Award zeichnet Da...
4,Bewerben bis 31.07.: Wiss. Mitarbeiter*in im P...,2026-07-16,Christian Wachter,Juli 2026,1,Stellenanzeigen,digital philosophy | Digitale Edition | Digita...,0,439,Am Philosophischen Seminar im Fachbereich 08 d...
5,Clustertreffen des N4O CC Data Capture and Cre...,2026-07-27,Anja Gerber,Juli 2026,1,Forschungsdaten | Forschungsinfrastruktur | Ve...,Digitale Sammlungen | Kulturdaten | kulturelle...,0,135,"Am\n8. September 2026, 14:30 bis 16:00 Uhr\n, ..."
6,Call for Hosts DHd2029,2026-07-29,Vivien Wolter,Juli 2026,1,Call | Konferenz,,0,495,"DHd20xx ist die jährliche, internationale Fach..."
7,Bericht vom Hands-On Lab “Digital Footprint – ...,2026-07-24,Ulrike Wuttke,Juli 2026,1,Allgemein | Open Access | Porträt (Projekt) | ...,Datentracking | OER | Science Tracking | Workshop,0,697,Autor*innen: Ulrike Wuttke (Fachhochschule Pot...
8,Lecture: Tackling Data Gaps in Digital Humanit...,2026-07-28,Polina Solonets,Juli 2026,1,Allgemein | Veranstaltungen | Vortrag,,0,280,We invite you to the online lecture by Dr. Sar...
9,Bewerben bis 10.08.2026: Akad. Rätin/Akad. Rat...,2026-07-21,Christian Wachter,Juli 2026,1,Stellenanzeigen,Digital History | Stelle | Stellenangebot,0,42,Am Historischen Seminar der Universität Münste...


In [6]:
print(
    "Missing titles:",
    df["title"].fillna("").str.strip().eq("").sum()
)

print(
    "Missing dates:",
    df["publication_date"].fillna("").str.strip().eq("").sum()
)

print(
    "Missing authors:",
    df["author"].fillna("").str.strip().eq("").sum()
)

print(
    "Empty texts:",
    df["text"].fillna("").str.strip().eq("").sum()
)

df["text_starts_with_title"] = df.apply(
    lambda row: (
        bool(row["title"])
        and bool(row["text"])
        and row["text"].strip().lower().startswith(
            row["title"].strip().lower()
        )
    ),
    axis=1
)

print(
    "Texts starting with title:",
    df["text_starts_with_title"].sum()
)

Missing titles: 0
Missing dates: 0
Missing authors: 0
Empty texts: 0
Texts starting with title: 0


In [7]:
display(post_errors_df)

,post_id,url,archive_month,archive_page,error
0,11593,https://dhd-blog.org/?p=11593,April 2019,1,Invalid IPv6 URL
1,1952,https://dhd-blog.org/?p=1952,Juli 2013,1,Invalid IPv6 URL


In [8]:
def safe_url_info(href):
    """
    Safely resolve and parse a URL.
    Returns (full_url, parsed_url).
    Malformed URLs return (None, None).
    """
    if not href:
        return None, None

    try:
        full_url = urljoin(BASE_URL, href)
        parsed_url = urlparse(full_url)
        return full_url, parsed_url

    except ValueError:
        return None, None


def extract_categories(article):
    """Extract categories and ignore malformed links."""

    categories = []

    for link in article.find_all("a", href=True):
        _, parsed = safe_url_info(link.get("href"))

        if parsed is None:
            continue

        rel_values = link.get("rel", [])

        if isinstance(rel_values, str):
            rel_values = rel_values.split()

        is_category = (
            "/category/" in parsed.path.lower()
            or "category" in rel_values
        )

        if is_category:
            value = clean_spacing(
                link.get_text(" ", strip=True)
            )

            if value:
                categories.append(value)

    for link in article.select(
        ".cat-links a, "
        ".post-categories a, "
        ".entry-categories a"
    ):
        value = clean_spacing(
            link.get_text(" ", strip=True)
        )

        if value:
            categories.append(value)

    return unique_clean_values(categories)


def extract_tags(article):
    """Extract tags and ignore malformed links."""

    tags = []

    for link in article.find_all("a", href=True):
        _, parsed = safe_url_info(link.get("href"))

        if parsed is None:
            continue

        rel_values = link.get("rel", [])

        if isinstance(rel_values, str):
            rel_values = rel_values.split()

        is_category = (
            "/category/" in parsed.path.lower()
            or "category" in rel_values
        )

        is_tag = (
            "/tag/" in parsed.path.lower()
            or (
                "tag" in rel_values
                and not is_category
            )
        )

        if is_tag and not is_category:
            value = clean_spacing(
                link.get_text(" ", strip=True)
            )

            if value:
                tags.append(value)

    for link in article.select(
        ".tags-links a, "
        ".post-tags a, "
        ".entry-tags a"
    ):
        _, parsed = safe_url_info(link.get("href"))

        if parsed is None:
            continue

        if "/category/" not in parsed.path.lower():
            value = clean_spacing(
                link.get_text(" ", strip=True)
            )

            if value:
                tags.append(value)

    return unique_clean_values(tags)


def extract_clean_article_text(article, title):
    """
    Extract only the blog-post body and safely remove metadata.
    """

    cleaned = BeautifulSoup(
        str(article),
        "lxml"
    )

    remove_selectors = [
        "script",
        "style",
        "noscript",
        "iframe",
        "form",
        "nav",
        "header",
        "footer",

        "h1.entry-title",
        "h2.entry-title",
        "h1 > a[rel='bookmark']",
        "h2 > a[rel='bookmark']",

        ".entry-meta",
        ".post-meta",
        ".postmetadata",
        ".post-meta-data",
        ".entry-footer",
        ".post-footer",
        ".meta",

        ".byline",
        ".author",
        ".vcard",
        ".date",
        ".published",

        ".cat-links",
        ".tags-links",
        ".post-categories",
        ".entry-categories",
        ".post-tags",
        ".entry-tags",

        ".comments-link",
        ".comment-count",
        ".sharedaddy",
        ".sd-sharing-enabled",
        ".share-buttons",
        ".social-share",
        ".navigation",
        ".post-navigation",
        ".comments-area",
        "#comments",
        "#respond",
        ".leave-comment",
        ".comment-respond",
        ".related-posts",
        ".jp-relatedposts",
        ".yarpp-related"
    ]

    for selector in remove_selectors:
        for element in cleaned.select(selector):
            element.decompose()

    metadata_links = cleaned.select(
        "a[rel~='category'], "
        "a[rel~='tag'], "
        ".cat-links a, "
        ".tags-links a, "
        ".post-categories a, "
        ".post-tags a"
    )

    for link in metadata_links:
        parent = link.parent

        if parent:
            parent_text = clean_spacing(
                parent.get_text(" ", strip=True)
            ).lower()

            metadata_terms = [
                "kategorie",
                "kategorien",
                "category",
                "categories",
                "schlagwort",
                "schlagwörter",
                "tags"
            ]

            if any(
                term in parent_text
                for term in metadata_terms
            ):
                parent.decompose()
            else:
                link.decompose()

    text = cleaned.get_text(
        separator="\n",
        strip=True
    )

    text = clean_spacing(text)

    if title and text.lower().startswith(title.lower()):
        text = text[len(title):].strip()

    text = re.sub(
        r"^\s*\d*\s*Veröffentlicht\s+von\s+.+?"
        r"\s+am\s+\d{1,2}\.\s+\w+\s+\d{4}\s*",
        "",
        text,
        flags=re.IGNORECASE
    )

    footer_patterns = [
        r"\n\s*Kategorien?\s*:.*$",
        r"\n\s*Categories?\s*:.*$",
        r"\n\s*Schlagwörter?\s*:.*$",
        r"\n\s*Tags?\s*:.*$",
        r"\n\s*Kommentar schreiben.*$",
        r"\n\s*Hinterlasse einen Kommentar.*$"
    ]

    for pattern in footer_patterns:
        text = re.sub(
            pattern,
            "",
            text,
            flags=re.IGNORECASE | re.DOTALL
        )

    return clean_spacing(text)


# Retry only the failed posts
failed_ids = ["11593", "1952"]

retry_rows = []
still_failed = []

for post_id in tqdm(
    failed_ids,
    desc="Retrying corrected posts"
):
    try:
        original_record = post_links_df.loc[
            post_links_df["post_id"].astype(str) == post_id
        ].iloc[0].to_dict()

        retry_rows.append(
            scrape_post(original_record)
        )

    except Exception as error:
        still_failed.append({
            "post_id": post_id,
            "error": str(error)
        })

    time.sleep(1)


# Add recovered posts to df
if retry_rows:
    retry_df = pd.DataFrame(retry_rows)

    df = pd.concat(
        [df, retry_df],
        ignore_index=True
    )

    df = (
        df.drop_duplicates(
            subset=["post_id"],
            keep="last"
        )
        .reset_index(drop=True)
    )


print("Recovered posts:", len(retry_rows))
print("Still failed:", len(still_failed))
print("Final posts:", len(df))
print("Unique post IDs:", df["post_id"].nunique())
print("Duplicate rows:", df["post_id"].duplicated().sum())

if still_failed:
    display(pd.DataFrame(still_failed))


Retrying corrected posts: 100%|██████████| 2/2 [00:02<00:00,  1.13s/it]

Recovered posts: 2
Still failed: 0
Final posts: 2752
Unique post IDs: 2752
Duplicate rows: 0


## 4. Validate and Export the Dataset


In [9]:
# ============================================================
# SAVE FINAL OUTPUT FILES
# ============================================================

dataset_csv_path = DATA_DIR / "dhd_blog_dataset.csv"
dataset_xlsx_path = DATA_DIR / "dhd_blog_dataset.xlsx"
post_errors_path = DATA_DIR / "dhd_blog_post_errors.csv"
post_links_path = DATA_DIR / "dhd_blog_all_post_links.csv"

df.to_csv(
    dataset_csv_path,
    index=False,
    encoding="utf-8-sig"
)

df.to_excel(
    dataset_xlsx_path,
    index=False
)

post_errors_df.to_csv(
    post_errors_path,
    index=False,
    encoding="utf-8-sig"
)

post_links_df.to_csv(
    post_links_path,
    index=False,
    encoding="utf-8-sig"
)

print("Files saved successfully.")
print(f"Dataset CSV: {dataset_csv_path.resolve()}")
print(f"Dataset Excel: {dataset_xlsx_path.resolve()}")
print(f"Post errors: {post_errors_path.resolve()}")
print(f"Post links: {post_links_path.resolve()}")


C:\Users\jiwani\AppData\Local\Temp\ipykernel_6220\173764296.py:16: UserWarning: Cell contents too long (36590), truncated to 32767 characters
  df.to_excel(
C:\Users\jiwani\AppData\Local\Temp\ipykernel_6220\173764296.py:16: UserWarning: Cell contents too long (48875), truncated to 32767 characters
  df.to_excel(
C:\Users\jiwani\AppData\Local\Temp\ipykernel_6220\173764296.py:16: UserWarning: Cell contents too long (34541), truncated to 32767 characters
  df.to_excel(
C:\Users\jiwani\AppData\Local\Temp\ipykernel_6220\173764296.py:16: UserWarning: Cell contents too long (32913), truncated to 32767 characters
  df.to_excel(


Files saved successfully.
Dataset CSV: C:\Users\jiwani\Documents\dhd-blog-analysis\data\dhd_blog_dataset.csv
Dataset Excel: C:\Users\jiwani\Documents\dhd-blog-analysis\data\dhd_blog_dataset.xlsx
Post errors: C:\Users\jiwani\Documents\dhd-blog-analysis\data\dhd_blog_post_errors.csv
Post links: C:\Users\jiwani\Documents\dhd-blog-analysis\data\dhd_blog_all_post_links.csv
